In [ ]:
!pip install -q pandas openpyxl spacy germansentiment "transformers<5" "tokenizers<0.21"
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import re
import math
from collections import Counter

import pandas as pd
import openpyxl
import spacy
from germansentiment import SentimentModel


# 1) Modelle laden

nlp = spacy.load("de_core_news_sm")
sentiment_model = SentimentModel()


# 2) Datensatz laden

FILEPATH = "alle Textausschnitte datensatz.xlsx"

wb = openpyxl.load_workbook(FILEPATH, data_only=True)
ws = wb[wb.sheetnames[0]]

row_pair = [c.value for c in ws[1]]
row_type = [c.value for c in ws[2]]
row_text = [c.value for c in ws[3]]

records = []

for idx, text in enumerate(row_text):
    if text is None:
        continue

    # Robuste Paarzuordnung: immer zwei Spalten = ein Textpaar
    pair_id = idx // 2 + 1
    text_type = str(row_type[idx]).strip() if row_type[idx] else ("Mensch" if idx % 2 == 0 else "KI")

    records.append({
        "pair_id": pair_id,
        "text_type": text_type,
        "text": str(text).strip()
    })

df = pd.DataFrame(records)


# 3) Hilfsfunktionen

def alpha_tokens(doc):
    """Nur alphabetische Tokens ohne Satzzeichen/Leerraum."""
    return [t for t in doc if t.is_alpha]

def content_tokens(doc):
    """Inhaltswörter ohne Stopwörter und Funktionswörter."""
    return [
        t for t in doc
        if t.is_alpha
        and not t.is_stop
        and t.pos_ in {"NOUN", "PROPN", "ADJ", "VERB", "ADV"}
    ]

def sentence_passive_heuristic(sent):
    """
    Sehr einfache Passiv-Heuristik für Deutsch:
    Satz gilt als potenziell passiv, wenn eine Form von 'werden'
    zusammen mit einem Partizip II auftritt.
    """
    has_werden = any(tok.lemma_.lower() == "werden" for tok in sent)
    has_participle = any(tok.tag_ == "VVPP" for tok in sent)
    return has_werden and has_participle

def safe_div(num, den):
    return num / den if den else math.nan

def top_content_lemmas(doc, top_n=10):
    """
    Häufigste Inhalts-Lemmata.
    """
    lemmas = [
        t.lemma_.lower()
        for t in content_tokens(doc)
        if len(t.lemma_) > 2
    ]
    return Counter(lemmas).most_common(top_n)

def sentence_sentiment_distribution(text, doc):
    """
    Sentiment auf Satzebene:
    pos / neg / neutral als Anteile.
    """
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    if not sentences:
        return {"sent_pos": math.nan, "sent_neg": math.nan, "sent_neu": math.nan}

    labels = sentiment_model.predict_sentiment(sentences)
    total = len(labels)

    return {
        "sent_pos": labels.count("positive") / total,
        "sent_neg": labels.count("negative") / total,
        "sent_neu": labels.count("neutral") / total,
    }

def compute_features(text):
    doc = nlp(text)

    toks = alpha_tokens(doc)
    sents = [s for s in doc.sents]
    content = content_tokens(doc)

    n_tokens = len(toks)
    n_sentences = len(sents)
    n_types = len(set(t.text.lower() for t in toks))
    n_content = len(content)
    n_adj = sum(1 for t in toks if t.pos_ == "ADJ")

    passive_sentences = sum(1 for s in sents if sentence_passive_heuristic(s))

    # Lexikalische Diversität
    ttr = safe_div(n_types, n_tokens)

    # Wortschatzdichte: Inhaltswörter / alle Wörter
    lexical_density = safe_div(n_content, n_tokens)

    # Satzlänge
    avg_tokens_per_sentence = safe_div(n_tokens, n_sentences)

    # Adjektivdichte
    adjective_density = safe_div(n_adj, n_tokens)

    # Passivanteil auf Satzebene
    passive_ratio = safe_div(passive_sentences, n_sentences)

    # Häufige Inhaltswörter
    top_lemmas = top_content_lemmas(doc, top_n=10)

    # Sentiment
    sentiment_stats = sentence_sentiment_distribution(text, doc)

    return {
        "n_tokens": n_tokens,
        "n_sentences": n_sentences,
        "avg_tokens_per_sentence": avg_tokens_per_sentence,
        "type_token_ratio": ttr,
        "lexical_density": lexical_density,
        "adjective_density": adjective_density,
        "passive_ratio": passive_ratio,
        "top_content_lemmas": top_lemmas,
        **sentiment_stats
    }


# 4) Features berechnen

feature_rows = []

for _, row in df.iterrows():
    feats = compute_features(row["text"])
    feature_rows.append({
        "pair_id": row["pair_id"],
        "text_type": row["text_type"],
        **feats
    })

features_df = pd.DataFrame(feature_rows)

# Schönere Anzeige
display_cols = [
    "pair_id",
    "text_type",
    "n_tokens",
    "n_sentences",
    "avg_tokens_per_sentence",
    "type_token_ratio",
    "lexical_density",
    "adjective_density",
    "passive_ratio",
    "sent_pos",
    "sent_neg",
    "sent_neu"
]

features_df[display_cols].round(3)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,pair_id,text_type,n_tokens,n_sentences,avg_tokens_per_sentence,type_token_ratio,lexical_density,adjective_density,passive_ratio,sent_pos,sent_neg,sent_neu
0,1,Mensch,295,14,21.071,0.600,0.417,0.071,0.071,0.000,0.000,1.000
1,1,KI,290,20,14.500,0.607,0.421,0.034,0.050,0.000,0.050,0.950
2,2,Mensch,443,18,24.611,0.542,0.327,0.056,0.056,0.000,0.000,1.000
3,2,KI,327,35,9.343,0.590,0.398,0.040,0.029,0.086,0.171,0.743
4,3,Mensch,96,3,32.000,0.823,0.458,0.073,0.333,0.000,0.000,1.000
5,3,KI,128,9,14.222,0.758,0.430,0.055,0.000,0.000,0.000,1.000
6,4,Mensch,190,6,31.667,0.684,0.326,0.021,0.167,0.000,0.000,1.000
7,4,KI,181,16,11.312,0.680,0.343,0.022,0.062,0.062,0.062,0.875


In [ ]:

# 5) Paarweise Differenzen KI minus Mensch

wide = features_df.pivot(index="pair_id", columns="text_type")

def get_diff(metric):
    return wide[(metric, "KI")] - wide[(metric, "Mensch")]

pair_compare = pd.DataFrame({
    "diff_tokens_KI_minus_Mensch": get_diff("n_tokens"),
    "diff_sentences_KI_minus_Mensch": get_diff("n_sentences"),
    "diff_avg_tokens_per_sentence_KI_minus_Mensch": get_diff("avg_tokens_per_sentence"),
    "diff_ttr_KI_minus_Mensch": get_diff("type_token_ratio"),
    "diff_lexical_density_KI_minus_Mensch": get_diff("lexical_density"),
    "diff_adjective_density_KI_minus_Mensch": get_diff("adjective_density"),
    "diff_passive_ratio_KI_minus_Mensch": get_diff("passive_ratio"),
    "diff_sent_pos_KI_minus_Mensch": get_diff("sent_pos"),
    "diff_sent_neg_KI_minus_Mensch": get_diff("sent_neg"),
    "diff_sent_neu_KI_minus_Mensch": get_diff("sent_neu"),
}).round(3)

pair_compare

,diff_tokens_KI_minus_Mensch,diff_sentences_KI_minus_Mensch,diff_avg_tokens_per_sentence_KI_minus_Mensch,diff_ttr_KI_minus_Mensch,diff_lexical_density_KI_minus_Mensch,diff_adjective_density_KI_minus_Mensch,diff_passive_ratio_KI_minus_Mensch,diff_sent_pos_KI_minus_Mensch,diff_sent_neg_KI_minus_Mensch,diff_sent_neu_KI_minus_Mensch
pair_id,,,,,,,,,,
1,-5,6,-6.571,0.007,0.004,-0.037,-0.021,0.000,0.050,-0.050
2,-116,17,-15.268,0.048,0.070,-0.017,-0.027,0.086,0.171,-0.257
3,32,6,-17.778,-0.065,-0.029,-0.018,-0.333,0.000,0.000,0.000
4,-9,10,-20.354,-0.005,0.016,0.001,-0.104,0.062,0.062,-0.125


In [ ]:

# 6) Automatische Kurzinterpretation pro Paar

def summarize_pair(pair_id):
    sub = features_df[features_df["pair_id"] == pair_id].set_index("text_type")

    m = sub.loc["Mensch"]
    k = sub.loc["KI"]

    lines = []
    lines.append(f"Textpaar {pair_id}:")

    # Länge / Struktur
    if k["avg_tokens_per_sentence"] < m["avg_tokens_per_sentence"]:
        lines.append(
            f"- Der KI-Text ist satzförmig stärker segmentiert "
            f"({k['avg_tokens_per_sentence']:.1f} vs. {m['avg_tokens_per_sentence']:.1f} Tokens pro Satz)."
        )
    else:
        lines.append(
            f"- Der menschliche Text ist stärker segmentiert "
            f"({m['avg_tokens_per_sentence']:.1f} vs. {k['avg_tokens_per_sentence']:.1f} Tokens pro Satz)."
        )

    # Lexikalität
    if k["type_token_ratio"] > m["type_token_ratio"]:
        lines.append(
            f"- Die lexikalische Diversität ist im KI-Text leicht höher "
            f"({k['type_token_ratio']:.3f} vs. {m['type_token_ratio']:.3f})."
        )
    elif k["type_token_ratio"] < m["type_token_ratio"]:
        lines.append(
            f"- Die lexikalische Diversität ist im menschlichen Text höher "
            f"({m['type_token_ratio']:.3f} vs. {k['type_token_ratio']:.3f})."
        )
    else:
        lines.append("- Die lexikalische Diversität ist nahezu identisch.")

    # Adjektive
    if k["adjective_density"] > m["adjective_density"]:
        lines.append("- Der KI-Text arbeitet relativ stärker mit Adjektiven.")
    elif k["adjective_density"] < m["adjective_density"]:
        lines.append("- Der menschliche Text arbeitet relativ stärker mit Adjektiven.")

    # Passiv
    if not math.isnan(k["passive_ratio"]) and not math.isnan(m["passive_ratio"]):
        if k["passive_ratio"] > m["passive_ratio"]:
            lines.append("- Die Passivheuristik fällt im KI-Text etwas höher aus.")
        elif k["passive_ratio"] < m["passive_ratio"]:
            lines.append("- Die Passivheuristik fällt im menschlichen Text etwas höher aus.")

    # Sentiment
    if not math.isnan(k["sent_pos"]) and not math.isnan(m["sent_pos"]):
        if k["sent_pos"] > m["sent_pos"]:
            lines.append("- Auf Satzebene wirkt der KI-Text tendenziell positiver.")
        elif k["sent_pos"] < m["sent_pos"]:
            lines.append("- Auf Satzebene wirkt der menschliche Text tendenziell positiver.")

    return "\n".join(lines)

for pid in sorted(features_df["pair_id"].unique()):
    print(summarize_pair(pid))
    print()

Textpaar 1:
- Der KI-Text ist satzförmig stärker segmentiert (14.5 vs. 21.1 Tokens pro Satz).
- Die lexikalische Diversität ist im KI-Text leicht höher (0.607 vs. 0.600).
- Der menschliche Text arbeitet relativ stärker mit Adjektiven.
- Die Passivheuristik fällt im menschlichen Text etwas höher aus.

Textpaar 2:
- Der KI-Text ist satzförmig stärker segmentiert (9.3 vs. 24.6 Tokens pro Satz).
- Die lexikalische Diversität ist im KI-Text leicht höher (0.590 vs. 0.542).
- Der menschliche Text arbeitet relativ stärker mit Adjektiven.
- Die Passivheuristik fällt im menschlichen Text etwas höher aus.
- Auf Satzebene wirkt der KI-Text tendenziell positiver.

Textpaar 3:
- Der KI-Text ist satzförmig stärker segmentiert (14.2 vs. 32.0 Tokens pro Satz).
- Die lexikalische Diversität ist im menschlichen Text höher (0.823 vs. 0.758).
- Der menschliche Text arbeitet relativ stärker mit Adjektiven.
- Die Passivheuristik fällt im menschlichen Text etwas höher aus.

Textpaar 4:
- Der KI-Text ist satzf